In [0]:
import sys
sys.path.append("../..")

In [0]:
import pyspark.sql.functions as F

import common.transformations as TR
from common.io import read_table, write_silver
from common.config import bronze_table, silver_table

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

PRODUCT_LINE_MAP = {
    "R": "Road",
    "T": "Touring",
    "M": "Mountain",
    "S": "Other Sales"
}

In [0]:
df = read_table(spark, bronze_table('crm_prd_info'))

In [0]:
df = TR.trim_string_columns(df)
df = TR.map_codes_to_labels(df, 'prd_line', PRODUCT_LINE_MAP)
df = TR.rename_columns(df, RENAME_MAP)
df = df.withColumn("cost", F.coalesce("cost", F.lit(0)))


In [0]:
write_silver(df, silver_table('crm_products'))

In [0]:
%sql
SELECT * FROM baraa_dev_project.silver.crm_products LIMIT 10